# GridLeak AI
## Model Eğitimi ve Hiperparametre Denemeleri

Durum: Başlangıç iskeleti. Veri analizi ve model eğitimi henüz yapılmadı.
Hücreleri sırayla çalıştıracağız; gerçek bulgular işlemden sonra yazılacak.


### Ortak deney düzeni

Aynı bölmeler, rastgelelik tohumu, seçim metriği ve çapraz doğrulama.


In [2]:
from pathlib import Path
import json
import pandas as pd

# Açık notebook'un bulunduğu konumdan proje klasörünü buluyoruz.
proje_klasoru = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "data" / "processed" /
            "egitim_ozellikleri_2016_v1.csv").is_file()
    ),
    None
)

if proje_klasoru is None:
    raise FileNotFoundError(
        "Eğitim dosyası bulunamadı. "
        "Bu notebook'u GridLeakAI/ai/notebooks içinden açın."
    )

processed_klasoru = proje_klasoru / "data" / "processed"

# 02 defterinde kaydettiğimiz eğitim tablosunu okuyoruz.
egitim = pd.read_csv(
    processed_klasoru / "egitim_ozellikleri_2016_v1.csv",
    dtype={"CONS_NO": str}
)

# Kaydedilmiş özellik adlarını ve sırasını okuyoruz.
ayarlar = json.loads(
    (processed_klasoru / "ozellik_tanimi_2016_v1.json")
    .read_text(encoding="utf-8")
)

ozellik_adlari = ayarlar["ozellikler"]

# Model girdileri ve doğru cevapları ayrı tutuyoruz.
X = egitim[ozellik_adlari].copy()
y = egitim["FLAG"].copy()

# Kimlik veya hedef yanlışlıkla model girdisine girmesin.
assert "CONS_NO" not in X.columns
assert "FLAG" not in X.columns
assert egitim["CONS_NO"].is_unique
assert y.notna().all() and set(y.unique()) == {0, 1}

print("Kullanılan proje:", proje_klasoru)
print("Eğitim abonesi:", len(X))
print("Özellik sayısı:", X.shape[1])

print("\nModelin kullanacağı özellikler:")
for ad in ozellik_adlari:
    print("-", ad)

print("\nEksik özellik değerleri:")
print(X.isna().sum())

Kullanılan proje: C:\Users\kadir\GridLeakAI
Eğitim abonesi: 33858
Özellik sayısı: 7

Modelin kullanacağı özellikler:
- ortalama_tuketim
- ortanca_tuketim
- tuketim_standart_sapma
- en_yuksek_tuketim
- sifir_gun_orani
- donemsel_degisim_yuzdesi
- en_uzun_sifir_serisi

Eksik özellik değerleri:
ortalama_tuketim               0
ortanca_tuketim                0
tuketim_standart_sapma         0
en_yuksek_tuketim              0
sifir_gun_orani                0
donemsel_degisim_yuzdesi    4050
en_uzun_sifir_serisi           0
dtype: int64


### Dengesizlik ve SMOTE

Dengelemesiz ve eğitim katında SMOTE deneyleri; test doğal dağılımda kalır.


In [5]:
from sklearn.model_selection import StratifiedKFold

# Aynı bölmeleri tekrar elde edebilmek için sabit sayı.
RASTGELELIK_TOHUMU = 42

# Sınıf oranlarını koruyan 5 parçalı doğrulama düzeni.
capraz_dogrulama = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RASTGELELIK_TOHUMU
)

# Dört modelde de aynı bölmeleri kullanacağız.
cv_bolmeleri = list(capraz_dogrulama.split(X, y))

bolme_ozeti = []

for tur, (ogrenme, dogrulama) in enumerate(cv_bolmeleri, start=1):
    bolme_ozeti.append({
        "Tur": tur,
        "Öğrenme sayısı": len(ogrenme),
        "Doğrulama sayısı": len(dogrulama),
        "Öğrenme kaçak (%)": y.iloc[ogrenme].mean() * 100,
        "Doğrulama kaçak (%)": y.iloc[dogrulama].mean() * 100
    })

display(pd.DataFrame(bolme_ozeti).round(2))

,Tur,Öğrenme sayısı,Doğrulama sayısı,Öğrenme kaçak (%),Doğrulama kaçak (%)
0,1,27086,6772,8.45,8.45
1,2,27086,6772,8.45,8.46
2,3,27086,6772,8.45,8.46
3,4,27087,6771,8.45,8.45
4,5,27087,6771,8.45,8.45


In [6]:
# Eğitim grubundaki her etiketten kaç abone olduğunu sayıyoruz.
sinif_sayilari = y.value_counts().sort_index()

# Sayıları ve yüzdeleri bir tabloda gösteriyoruz.
sinif_ozeti = pd.DataFrame({
    "Abone sayısı": sinif_sayilari,
    "Oran (%)": sinif_sayilari / len(y) * 100
})

sinif_ozeti.index.name = "FLAG"

display(sinif_ozeti.round(2))

# Kaçak etiketli bir aboneye karşılık kaç normal etiketli
# abone bulunduğunu hesaplıyoruz.
dengesizlik_orani = sinif_sayilari.loc[0] / sinif_sayilari.loc[1]

print(
    "Her 1 kaçak etiketli aboneye karşılık",
    round(dengesizlik_orani, 2),
    "kaçak olmayan etiketli abone var."
)

,Abone sayısı,Oran (%)
FLAG,,
0,30996,91.55
1,2862,8.45


Her 1 kaçak etiketli aboneye karşılık 10.83 kaçak olmayan etiketli abone var.


In [10]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline


# Her model için yeni bir hazırlık zinciri oluşturur.
def egitim_zinciri_olustur(model, olcekle=True, smote_kullan=False):
    adimlar = []

    # Eksik özellikleri öğrenme grubunun ortancasıyla doldurur.
    adimlar.append((
        "eksik_doldurma",
        SimpleImputer(strategy="median")
    ))

    # İstenirse özellikleri standartlaştırır.
    if olcekle:
        adimlar.append((
            "olcekleme",
            StandardScaler()
        ))

    # İstenirse sentetik azınlık sınıfı örnekleri üretir.
    if smote_kullan:
        adimlar.append((
            "smote",
            SMOTE(
                random_state=RASTGELELIK_TOHUMU,
                k_neighbors=5
            )
        ))

    # Öğrenme işlemini yapacak modeli sona ekler.
    adimlar.append(("model", model))

    return Pipeline(adimlar)


print("Eğitim zincirini oluşturacak fonksiyon hazır.")
print("Henüz veri dönüştürülmedi veya model eğitilmedi.")

Eğitim zincirini oluşturacak fonksiyon hazır.
Henüz veri dönüştürülmedi veya model eğitilmedi.


### Logistic Regression

Eğitim ve hiperparametre denemeleri burada yapılacak.


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Eğitilecek modeli tanımlıyoruz.
lr_model = LogisticRegression(
    max_iter=3000,
    random_state=RASTGELELIK_TOHUMU
)

# Eksik doldurma + ölçekleme + SMOTE + model zinciri.
lr_zinciri = egitim_zinciri_olustur(
    model=lr_model,
    olcekle=True,
    smote_kullan=True
)

# SMOTE kapalı ve açık olacak şekilde farklı ayarları deniyoruz.
lr_parametreleri = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "smote": [
        "passthrough",
        SMOTE(
            random_state=RASTGELELIK_TOHUMU,
            k_neighbors=5
        )
    ]
}

# Her ayarı daha önce hazırladığımız aynı 5 bölmede değerlendiriyoruz.
lr_arama = GridSearchCV(
    estimator=lr_zinciri,
    param_grid=lr_parametreleri,
    scoring={
        "ap": "average_precision",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    refit="ap",
    cv=cv_bolmeleri,
    n_jobs=1,
    verbose=1,
    return_train_score=True,
    error_score="raise"
)

# GERÇEK EĞİTİM BU SATIRDA BAŞLAR.
lr_arama.fit(X, y)

print("Logistic Regression denemeleri tamamlandı.")
print("Seçilen C değeri:", lr_arama.best_params_["model__C"])

secilen_smote = lr_arama.best_params_["smote"]
print(
    "Seçilen dengeleme:",
    "SMOTE yok" if secilen_smote == "passthrough" else "SMOTE var"
)

print(
    "En iyi ortalama doğrulama AP skoru:",
    round(lr_arama.best_score_, 4)
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Logistic Regression denemeleri tamamlandı.
Seçilen C değeri: 0.01
Seçilen dengeleme: SMOTE var
En iyi ortalama doğrulama AP skoru: 0.2231


In [12]:
# GridSearchCV'nin bütün deneme sonuçlarını tabloya alıyoruz.
lr_sonuclar = pd.DataFrame(lr_arama.cv_results_)

# Her satır bir ayar kombinasyonunun 5 turluk özetidir.
lr_karsilastirma = pd.DataFrame({
    "C": lr_sonuclar["param_model__C"].astype(float),

    "Dengeleme": [
        "SMOTE yok" if isinstance(secenek, str) else "SMOTE var"
        for secenek in lr_sonuclar["param_smote"]
    ],

    "Eğitim AP": lr_sonuclar["mean_train_ap"],
    "Doğrulama AP": lr_sonuclar["mean_test_ap"],
    "AP standart sapma": lr_sonuclar["std_test_ap"],
    "Precision": lr_sonuclar["mean_test_precision"],
    "Recall": lr_sonuclar["mean_test_recall"],
    "F1": lr_sonuclar["mean_test_f1"]
})

# En yüksek doğrulama AP skorunu üstte gösteriyoruz.
lr_karsilastirma = lr_karsilastirma.sort_values(
    "Doğrulama AP",
    ascending=False
).reset_index(drop=True)

display(lr_karsilastirma.round(4))

# Seçilen, eğitim grubunun tamamında yeniden eğitilmiş zincir.
en_iyi_lr = lr_arama.best_estimator_

,C,Dengeleme,Eğitim AP,Doğrulama AP,AP standart sapma,Precision,Recall,F1
0,0.01,SMOTE var,0.2226,0.2231,0.0070,0.1627,0.5203,0.2477
1,0.10,SMOTE var,0.2237,0.2225,0.0060,0.1663,0.5101,0.2508
2,1.00,SMOTE var,0.2225,0.2220,0.0056,0.1648,0.5052,0.2485
3,10.00,SMOTE var,0.2213,0.2209,0.0064,0.1635,0.5024,0.2467
4,0.10,SMOTE yok,0.2183,0.2185,0.0062,0.6101,0.0339,0.0640
5,1.00,SMOTE yok,0.2171,0.2170,0.0063,0.6005,0.0367,0.0689
6,0.01,SMOTE yok,0.2146,0.2151,0.0076,0.6062,0.0241,0.0461
7,10.00,SMOTE yok,0.2153,0.2151,0.0064,0.5812,0.0367,0.0688


### Random Forest

Eğitim ve hiperparametre denemeleri burada yapılacak.


In [14]:
from sklearn.ensemble import RandomForestClassifier

# Birden fazla karar ağacından oluşan modeli tanımlıyoruz.
rf_model = RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=2,
    random_state=RASTGELELIK_TOHUMU,
    n_jobs=-1
)

# SMOTE yakın komşuları kullandığı için özellikleri ölçekliyoruz.
# Random Forest'ın kendisi ölçeklemeye ihtiyaç duymaz.
rf_zinciri = egitim_zinciri_olustur(
    model=rf_model,
    olcekle=True,
    smote_kullan=True
)

# İki ağaç derinliği ve iki dengeleme seçeneği deniyoruz.
rf_parametreleri = {
    "model__max_depth": [10, None],
    "smote": [
        "passthrough",
        SMOTE(
            random_state=RASTGELELIK_TOHUMU,
            k_neighbors=5
        )
    ]
}

# Logistic Regression ile aynı değerlendirme düzeni.
rf_arama = GridSearchCV(
    estimator=rf_zinciri,
    param_grid=rf_parametreleri,
    scoring={
        "ap": "average_precision",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    refit="ap",
    cv=cv_bolmeleri,
    n_jobs=1,
    verbose=1,
    return_train_score=True,
    error_score="raise"
)

# Random Forest eğitimi burada başlıyor.
rf_arama.fit(X, y)

# Seçilen, eğitim grubunun tamamında yeniden eğitilmiş zincir.
en_iyi_rf = rf_arama.best_estimator_

print("Random Forest denemeleri tamamlandı.")
print(
    "Seçilen maksimum derinlik:",
    rf_arama.best_params_["model__max_depth"]
)

print(
    "Seçilen dengeleme:",
    "SMOTE yok"
    if rf_arama.best_params_["smote"] == "passthrough"
    else "SMOTE var"
)

print(
    "En iyi ortalama doğrulama AP skoru:",
    round(rf_arama.best_score_, 4)
)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Random Forest denemeleri tamamlandı.
Seçilen maksimum derinlik: 10
Seçilen dengeleme: SMOTE yok
En iyi ortalama doğrulama AP skoru: 0.2393


### SVM

Eğitim ve hiperparametre denemeleri burada yapılacak.


### XGBoost

Eğitim ve hiperparametre denemeleri burada yapılacak.


### Model seçimi ve kayıt

CV sonuçlarına göre seçim; model ve dönüşümler ai/artifacts içine.
